# 🎯 Is the difference real, and is it big?

**Module 1 · Session 03, part 2**

Same brief. The team wants to know what makes a track popular, and part 1 came back with an
uncomfortable answer: no audio measurement correlates with popularity above 0.14.

That is a correlation, though, and "no relationship" is a claim you have to be able to defend.
Today you test it, and you test the three other things in the file that might move popularity
instead.

**Four questions. Answer all four in the next cell before you run anything else.** You get them
back at the end with the real numbers beside them.

1. A track that sits on three or more playlists, against a track on one. How many points apart
   on the 0-100 popularity score?
2. Tracks between 2½ and 3 minutes, against tracks over 5 minutes. How many points?
3. All ten audio measurements together. What share of the variation in popularity do they
   account for between them?
4. Nine percent of tracks score exactly 0. Are those spread evenly across the six genres?

Guess properly. A number you committed to is much harder to forget than one you read.


In [ ]:
# Overwrite these with your own answers. What is here is roughly what a room says.
guess = {
    "popularity gap, 3+ playlists vs 1": 8,
    "popularity gap, short vs long tracks": 3,
    "% of popularity explained by audio": 45,
    "zeros spread evenly across genres?": "yes",
}


In [ ]:
# Setup. Same file as last week, plus one style block so every chart here matches.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import HTML
from scipy import stats

# One theme for the whole notebook, so no chart below needs styling of its own.
# The colours are chosen to stay distinguishable with colour-blindness.
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#8b8a85"
sns.set_theme(style="whitegrid", palette=[BLUE, ORANGE, GREY],
              rc={"figure.dpi": 110, "grid.color": "#ececea", "font.size": 10,
                  "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left"})

URL = "https://raw.githubusercontent.com/aaubs/ds-master/codex/m1-pandas-2026/data/M1_2026/spotify_songs.csv"

songs = pd.read_csv(URL).rename(columns={
    "track_name": "title", "track_artist": "artist",
    "track_popularity": "popularity", "playlist_genre": "genre"})
tracks = songs.drop_duplicates("track_id").copy()

# How many playlists is each track on? One line, and it becomes the first question.
tracks["placements"] = tracks["track_id"].map(songs["track_id"].value_counts())


def listen(rows, *extra_columns):
    """Show rows with a clickable Spotify link. track_id is a real Spotify ID."""
    out = rows[["title", "artist", *extra_columns]].round(3)
    out["listen"] = "https://open.spotify.com/track/" + rows["track_id"]
    return HTML(out.to_html(render_links=True, escape=False, index=False))


print(f"Distinct songs: {len(tracks):,}")
print(f"On more than one playlist: {(tracks['placements'] > 1).sum():,}")


Each question has a test behind it, and picking the test is the easy part.

| Your question | The test |
|---|---|
| Do two groups differ on a number? | t-test |
| Do three or more groups differ on a number? | one-way ANOVA |
| Are two categorical columns related? | chi-square |
| Do two numbers move together? | correlation, which was part 1 |

All of them answer whether a difference could be chance. None answers whether it is big enough
to act on, and none answers whether it is a difference you can do anything about. Both of those
turn up in the next twenty minutes.

## 📊 Question 1. Playlist placements

This is the comparison you just handed in. Question 3 of the assignment asked whether tracks
that get around look different, and left you to pick the cut-off. Here is what it looks like with
the machinery from today.


In [ ]:
by_placements = tracks.groupby(tracks["placements"].clip(upper=5))["popularity"].agg(["mean", "count"])

fig, ax = plt.subplots(figsize=(7, 3.2))
sns.barplot(x=by_placements.index, y=by_placements["mean"], color=BLUE, ax=ax)
ax.bar_label(ax.containers[0], fmt="{:.0f}".format, padding=3)
ax.set(title="Mean popularity by number of playlists the track is on",
       xlabel="playlists (5 means five or more)", ylabel="popularity", ylim=(0, 95))
plt.show()

display(by_placements.round(1))


From 37 to 85. Nothing else in this file moves popularity like that.


In [ ]:
one = tracks.loc[tracks["placements"] == 1, "popularity"]
many = tracks.loc[tracks["placements"] >= 3, "popularity"]

# equal_var=False is Welch's version, which does not assume equal spread. Use it by default.
test = stats.ttest_ind(many, one, equal_var=False)
print(f"one playlist:      {len(one):,} tracks, mean {one.mean():.1f}")
print(f"three or more:     {len(many):,} tracks, mean {many.mean():.1f}")
print(f"gap:               {many.mean() - one.mean():.1f} points")
print(f"p value:           {test.pvalue:.1e}")


A 34-point gap on a 0-100 scale, at p = 10^-254. By every statistical standard this is the
finding of the day.

> ⚠️ **And it is useless to the brief.** Nobody chooses to be on more playlists. Editors add
> tracks that are already doing well, and being added then pushes them further. The arrow runs
> both ways and this table cannot separate them.
>
> Hand this to the team as "get on more playlists and popularity goes up by 34 points" and you
> have told them to make the thermometer read higher by holding a match to it.

The strongest effect in a dataset is very often the outcome wearing a different hat. Ask what
would have to be true for the relationship to run the way you are about to claim, before you
claim it.

## Effect size

The gap is 34 points. Whether that is large depends on how spread out popularity was to begin
with, and Cohen's d is the usual way to say so.


In [ ]:
def cohens_d(a, b):
    """Difference in means, measured in pooled standard deviations."""
    return (a.mean() - b.mean()) / np.sqrt((a.std() ** 2 + b.std() ** 2) / 2)


print(f"3+ playlists against 1: d = {cohens_d(many, one):.2f}")


1.62. The rough labels are 0.2 small, 0.5 medium, 0.8 large, so this is off the end of the
scale that gets used in practice.

Correct, enormous, and not actionable. Keep all three of those in the same sentence.

## 📊 Question 2. Track length

Here is one you can act on. Length is decided before a track is released.


In [ ]:
minutes = tracks["duration_ms"] / 60_000
bands = pd.cut(minutes, [0, 2.5, 3, 3.5, 4, 5, 20],
               labels=["under 2:30", "2:30-3:00", "3:00-3:30", "3:30-4:00", "4:00-5:00", "over 5:00"])

by_length = tracks.groupby(bands, observed=True)["popularity"].agg(["mean", "count"])

fig, ax = plt.subplots(figsize=(7.5, 3.2))
sns.barplot(x=by_length.index, y=by_length["mean"], color=BLUE, ax=ax)
ax.bar_label(ax.containers[0], fmt="{:.1f}".format, padding=3)
ax.set(title="Mean popularity by track length", xlabel="", ylabel="popularity", ylim=(0, 52))
plt.show()

short = tracks.loc[tracks["duration_ms"].between(150_000, 180_000), "popularity"]
long = tracks.loc[tracks["duration_ms"] > 300_000, "popularity"]
print(f"2:30-3:00:  {len(short):,} tracks, mean {short.mean():.1f}")
print(f"over 5:00:  {len(long):,} tracks, mean {long.mean():.1f}")
print(f"gap: {short.mean() - long.mean():.1f} points, d = {cohens_d(short, long):.2f}")
print(f"p value: {stats.ttest_ind(short, long, equal_var=False).pvalue:.1e}")


Eleven points, d = 0.50, and the bars slope one way across the whole range. This one a team can
use, because length is a decision somebody makes in a studio.

It is also a quarter of the effect that placements had, which is the usual shape of things: the
levers you can pull are smaller than the ones you cannot.

> 🧭 **Judgement call.** Ask an agent whether track length affects popularity and you get a
> correct test, a correct p value and a verdict. Whether the effect is one anybody can act on
> was not in the request and is not in the output. On business-sized data that missing half is
> usually the entire decision.

## When significant means nothing at all

Two genres that are almost identical on popularity.


In [ ]:
rock = tracks.loc[tracks["genre"] == "rock", "popularity"]
latin = tracks.loc[tracks["genre"] == "latin", "popularity"]

print(f"rock:  {len(rock):,} tracks, mean {rock.mean():.2f}")
print(f"latin: {len(latin):,} tracks, mean {latin.mean():.2f}")
print(f"gap:   {rock.mean() - latin.mean():.2f} points, d = {cohens_d(rock, latin):.3f}")
print(f"p value: {stats.ttest_ind(rock, latin, equal_var=False).pvalue:.4f}")
print("Significant at 5 percent?", stats.ttest_ind(rock, latin, equal_var=False).pvalue < 0.05)


Under two points on a 0-100 index, d = 0.07, and significant at better than one in a thousand.

So the word "significant" now covers a 34-point gap and a 1.75-point gap, and a paper reporting
p < 0.05 for the second would be reporting it accurately.

## Why sample size did all of that

Every group here has around four thousand rows. Take the comparison you just saw and run it on
thirty rows a side instead, two hundred times.


In [ ]:
p_values = pd.Series([
    stats.ttest_ind(rock.sample(30, random_state=seed),
                    latin.sample(30, random_state=seed + 999), equal_var=False).pvalue
    for seed in range(200)
])

fig, ax = plt.subplots(figsize=(8, 3.4))
sns.histplot(x=p_values, bins=20, color=BLUE, ax=ax)
ax.axvline(0.05, color=ORANGE, linewidth=2)
ax.text(0.062, ax.get_ylim()[1] * 0.85, "0.05", color=ORANGE)
ax.set(title="p values from 200 runs on 30 rows a side", xlabel="p value", ylabel="runs")
plt.show()

print(f"median p: {p_values.median():.3f}")
print(f"under 0.05 in {(p_values < 0.05).mean():.1%} of runs")


The difference in the data never changed. Only the row count did.

A p value reports on a difference *and* on your sample size together, so with enough rows any
difference that is not exactly zero will eventually cross any threshold you pick. On a table
this size, significance is close to free.

## 🎤 Break: ten names

Groups of three. Five minutes. No phones.

Write down Spotify's ten most-streamed artists worldwide in 2025, in any order. A point each.


In [ ]:
top10_2025 = ["Bad Bunny", "Taylor Swift", "The Weeknd", "Drake", "Billie Eilish",
              "Kendrick Lamar", "Bruno Mars", "Ariana Grande", "Arijit Singh", "Fuerza Regida"]

# How many rows does each of them have in our 2020 file?
rows = [songs["artist"].str.contains(name, case=False, na=False).sum() for name in top10_2025]
display(pd.Series(rows, index=top10_2025, name="rows in our file").to_frame())


Eight of the ten are in the file, some heavily. Arijit Singh, a Hindi playback singer and the
first Indian artist in the global top ten, has nothing. Fuerza Regida, a regional Mexican band
from San Bernardino, has nothing.

Those two are also the two nobody in the room wrote down.

Six genres, one market, 2020. When this notebook says "genre" it means six categories somebody
at Spotify drew for an audience that looks a lot like this classroom, and that is the honest
limit on every number in it.

## 📊 Question 3. How much does anything explain?

Two ways of asking the same thing. ANOVA and eta squared do it for a category; for the ten
audio columns at once you need something you meet properly later, which is fine to preview.


In [ ]:
by_genre = tracks.groupby("genre")["popularity"]
anova = stats.f_oneway(*[group for _, group in by_genre])

grand_mean = tracks["popularity"].mean()
between = (by_genre.size() * (by_genre.mean() - grand_mean) ** 2).sum()
total = ((tracks["popularity"] - grand_mean) ** 2).sum()
eta_squared = between / total

print(f"genre: F = {anova.statistic:.0f}, p = {anova.pvalue:.1e}, eta squared = {eta_squared:.3f}")


In [ ]:
# A preview of regression: fit all ten audio columns at once and ask how much they account for.
audio = ["danceability", "energy", "loudness", "speechiness", "acousticness",
         "instrumentalness", "liveness", "valence", "tempo", "duration_ms"]

design = np.c_[np.ones(len(tracks)), tracks[audio].to_numpy()]
coefficients, *_ = np.linalg.lstsq(design, tracks["popularity"].to_numpy(), rcond=None)
predicted = design @ coefficients
r_squared = 1 - ((tracks["popularity"] - predicted) ** 2).sum() / total

print(f"all ten audio measurements together: R squared = {r_squared:.3f}")


All ten audio measurements together account for 6 percent of the variation in popularity.
Genre on its own accounts for 4 percent.

Ninety-four percent of what makes a track popular is not in this file. Not in the audio model,
not in the genre label, not in anything Spotify measured about the recording. Marketing,
timing, an editor's decision, a film soundtrack, a video that went round — none of it is here.

That is the real answer to the brief, and one honest sentence beats any model you could fit to
these columns.

## 📊 Question 4. Are the zeros spread evenly?

Part 1 found the spike: nine percent of tracks score exactly 0, and we could not tell whether
that means unlistened or unmeasured. Chi-square tests whether those zeros fall evenly across
genres, which is a data-quality question rather than a musical one.


In [ ]:
zeros = pd.crosstab(tracks["genre"], tracks["popularity"].eq(0))
zeros.columns = ["has a score", "exactly zero"]
share = (zeros["exactly zero"] / zeros.sum(axis=1)).sort_values()

fig, ax = plt.subplots(figsize=(7, 3.2))
sns.barplot(x=share.values, y=share.index, color=BLUE, ax=ax)
ax.bar_label(ax.containers[0], fmt="{:.1%}".format, padding=4)
ax.set(title="Share of tracks scoring exactly zero", xlabel="", ylabel="", xlim=(0, 0.17))
ax.set_xticks([])
sns.despine(bottom=True)
plt.show()

chi2, p_value, dof, expected = stats.chi2_contingency(zeros)
print(f"chi-square: {chi2:.0f}, p: {p_value:.1e}")
print(f"Cramer's V: {np.sqrt(chi2 / (zeros.to_numpy().sum() * (min(zeros.shape) - 1))):.3f}")


No. edm carries more than twice the share that pop does, 13.3 percent against 6.1, and
chi-square puts the odds of that being chance at one in 10^37.

Which turns the part 1 judgement call into something with a number on it.


In [ ]:
kept = tracks[tracks["popularity"] > 0]

comparison = pd.DataFrame({
    "with zeros": tracks.groupby("genre")["popularity"].mean(),
    "without zeros": kept.groupby("genre")["popularity"].mean(),
}).round(1)
comparison["shift"] = (comparison["without zeros"] - comparison["with zeros"]).round(1)
display(comparison.sort_values("with zeros"))


Every genre moves up, and edm moves furthest, because edm had the most zeros. A decision you
made about data quality has quietly re-ranked the thing the team asked you about.

The ranking happens to survive here. It did not have to, and you would not know unless you ran
both. Run both, report the one you chose, and say what the other one did.

## Before you go hunting

Run twenty tests on data with nothing in it and about one comes back under 0.05. That is what
the threshold means: the rate at which you expect to be fooled.

It used to be a slow problem, because twenty tests took an afternoon. An agent runs fifty
comparisons in one call and hands back the three that cleared 0.05, and those three are more
likely to be the noise than the finding.

Test the question you arrived with. If you do go fishing across every pair of columns, say so,
and treat whatever surfaces as something to check against other data.

## How to report it

Three things, in this order, and never the third on its own:

1. the size of the difference, in the units you measured
2. how many observations it rests on
3. the p value

"Tracks of 2½ to 3 minutes average 42.6 against 31.2 for tracks over five minutes, across 3,858
and 2,973 tracks, a gap of 11 points and d = 0.50 (p < 0.001)" is a sentence somebody can argue
with. "Track length was a significant predictor" is not.

## Your four answers


In [ ]:
truth = {
    "popularity gap, 3+ playlists vs 1": round(many.mean() - one.mean()),
    "popularity gap, short vs long tracks": round(short.mean() - long.mean()),
    "% of popularity explained by audio": round(r_squared * 100),
    "zeros spread evenly across genres?": "yes" if p_value > 0.05 else "no",
}
display(pd.DataFrame({"you said": guess, "the file says": truth}))


The two most rooms get wrong are 1 and 3, and in opposite directions. Placements matter far
more than anyone expects and cannot be used; the audio measurements matter far less than anyone
expects and are the only thing the brief asked about.

## ✍️ Your turn

Pick a comparison in this file you would actually want the answer to. Any two groups, any
column. Report the counts, the size of the gap, the effect size, then the p value.

Then write the one sentence you would send the team, and one saying what it does not tell them.


In [ ]:
# Your comparison.


### One answer: does releasing recently help?


In [ ]:
tracks["year"] = pd.to_datetime(tracks["track_album_release_date"],
                                format="mixed", errors="coerce").dt.year

era = pd.cut(tracks["year"], [1950, 2000, 2010, 2017, 2020],
             labels=["before 2000", "2000s", "2010-2017", "2018-2020"])
display(tracks.groupby(era, observed=True)["popularity"].agg(["mean", "count"]).round(1))

recent = tracks.loc[tracks["year"] >= 2018, "popularity"]
middle = tracks.loc[tracks["year"].between(2000, 2010), "popularity"]
print(f"gap: {recent.mean() - middle.mean():.1f} points, d = {cohens_d(recent, middle):.2f}")
print(f"p value: {stats.ttest_ind(recent, middle, equal_var=False).pvalue:.1e}")


Recent tracks score 14 points above tracks from the 2000s, d = 0.60. But look at the first
row: tracks from before 2000 score higher than either of the middle bands.

Age does not run in one direction, because a 1975 track still sitting on a 2020 playlist is
there for a reason. You are not looking at how popularity decays, you are looking at what
survived, and the two are easy to confuse.

The sentence for the team: recent releases score 14 points above 2000s releases, across 10,949
and 4,135 tracks. What it does not tell them: anything about how a new track will
age, because every old track in this file was selected for still being played.

## 📋 The memo

If you had to send one paragraph upstairs:

> Nothing we measure about the audio predicts popularity. Ten measurements together account for
> six percent of the variation; genre on its own, four. The strongest signal in the data is playlist
> placement, which is downstream of popularity rather than upstream, so it cannot be used as a
> lever. Track length is the one usable finding: under three minutes outperforms over five by
> about eleven points. Nine percent of the popularity column is a spike at zero of unclear
> meaning, and the ranking here holds whether we keep it or drop it.

## 📋 Takeaways

- A p value answers whether a difference exists given your sample size. It says nothing about
  size, and nothing about which way the arrow points.
- On business-sized tables nearly everything is significant, so effect size carries the finding.
- The biggest effect in a dataset is often the outcome in disguise. Ask what would have to be
  true for your causal story to hold.
- Report size, then counts, then the p value.
- Twenty tests on nothing produce roughly one significant result.
- All of this describes one 2020 playlist snapshot, and the two artists the room could not name
  are the two the file does not contain.
